# 11 · Bayesian Probabilistic Forecasting

Bayesian methods treat the unknown parameters $\theta$ as random. They produce the **posterior predictive
distribution**, which integrates over parameter uncertainty instead of plugging in a single estimate:
$$p(y_{T+h}\mid\mathcal D)=\int p(y_{T+h}\mid\theta,\mathcal D)\;p(\theta\mid\mathcal D)\,d\theta,\qquad
p(\theta\mid\mathcal D)\propto p(\mathcal D\mid\theta)\,p(\theta).$$

## Theory

### Epistemic + aleatoric uncertainty (law of total variance)
$$\text{Var}(y^*)=\underbrace{\mathbb E_{\theta}\big[\text{Var}(y^*\mid\theta)\big]}_{\text{aleatoric (noise)}}+\underbrace{\text{Var}_{\theta}\big(\mathbb E[y^*\mid\theta]\big)}_{\text{epistemic (parameters)}}$$
"Plug-in" forecasts ($\theta=\hat\theta$) keep only the first term, so they are **over-confident**, especially with
little data or many parameters (12 monthly seasonal effects need many years of data!).

### Toolbox
| Method | Notes |
|---|---|
| Conjugate Bayesian linear regression (Normal–Inverse-Gamma) | closed form, Student-t predictive. Used below |
| Dynamic linear models / Bayesian structural time series (BSTS) | Kalman filter + priors on variances. Spike-and-slab for regressors |
| Gaussian processes | a prior over *functions*. Kernels encode trend, seasonality and smoothness. Used below |
| Prophet | trend + Fourier seasonality. MAP by default, full MCMC with `mcmc_samples>0` |
| PyMC / NumPyro / Stan | any custom model, sampled with MCMC/NUTS or variational inference |
| Hierarchical (partial pooling) | share seasonality across many short series: a big win for new products |

### Conjugate Normal–Inverse-Gamma regression
Prior: $\beta\mid\sigma^2\sim N(m_0,\sigma^2V_0)$, $\sigma^2\sim\text{IG}(a_0,b_0)$. Posterior:
$V_n=(V_0^{-1}+X^\top X)^{-1}$, $m_n=V_n(V_0^{-1}m_0+X^\top y)$, $a_n=a_0+n/2$,
$b_n=b_0+\tfrac12(y^\top y+m_0^\top V_0^{-1}m_0-m_n^\top V_n^{-1}m_n)$.
Predictive for a new $x_*$: **Student-t** with $2a_n$ d.o.f., location $x_*^\top m_n$, scale$^2$ $=\frac{b_n}{a_n}(1+x_*^\top V_nx_*)$.
The term $x_*^\top V_nx_*$ is the epistemic part. It is larger for extrapolation (far-future trend values).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats, optimize
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(11)

def nig_posterior(X, y, m0, V0, a0, b0):
    V0i = np.linalg.inv(V0)
    Vn = np.linalg.inv(V0i + X.T @ X)
    mn = Vn @ (V0i @ m0 + X.T @ y)
    an = a0 + len(y) / 2
    bn = b0 + 0.5 * (y @ y + m0 @ V0i @ m0 - mn @ np.linalg.inv(Vn) @ mn)
    return mn, Vn, an, bn

def nig_predictive(Xs, mn, Vn, an, bn):
    scale = np.sqrt(bn / an * (1 + np.einsum("ij,jk,ik->i", Xs, Vn, Xs)))
    return stats.t(df=2 * an, loc=Xs @ mn, scale=scale)

## 1. Simulation study: when does parameter uncertainty matter?
A known monthly process: $\log y_t=5+0.01t+0.1\sin(2\pi t/12)+0.08\cos(2\pi t/12)+\varepsilon_t$, $\varepsilon_t\sim N(0,0.05^2)$.
We fit trend + 11 month dummies (13 parameters) on $n$ months and forecast the next 12. Then we check the
**coverage of 90% intervals** over 1 000 replications for

* **Plug-in Gaussian**: $\hat y\pm1.645\,\hat\sigma$ (residual s.d., the "naive" practice)
* **Bayesian predictive** (vague prior): Student-t with the leverage term

In [2]:
def design(n_total):
    idx = pd.date_range("2000-01-01", periods=n_total, freq="MS")
    return pf.seasonal_design(idx, "2000-01-01").to_numpy()

p = 13
# Reference prior p(beta, sigma^2) ∝ 1/sigma^2, written as a limit of the NIG family: V0 -> inf, a0 = -p/2, b0 = 0.
# Two traps (try them!):
#  * V0 = 100*I looks vague, but the prior variance is sigma^2*V0: with sigma ~ 0.05 that is a prior sd of 0.5 around 0
#    for an intercept of ~5. The prior-data conflict inflates sigma^2, so the intervals are far too wide.
#  * V0 = 1e6*I with a0 = b0 ≈ 0 keeps a sigma^(-p) factor, so the posterior has n instead of n - p degrees of freedom
#    and sigma^2 is under-estimated, so the intervals are too narrow when n is small.
m0, V0, a0, b0 = np.zeros(p), np.eye(p) * 1e8, -p / 2, 0.0
rows = []
for n in [18, 24, 36, 60, 120]:
    Xall = design(n + 12); tt = np.arange(n + 12)
    cov_plug, cov_bayes, w_plug, w_bayes = [], [], [], []
    for _ in range(1000):
        z = 5 + 0.01 * tt + 0.1 * np.sin(2 * np.pi * tt / 12) + 0.08 * np.cos(2 * np.pi * tt / 12) + rng.normal(0, 0.05, n + 12)
        X, Xs, yf = Xall[:n], Xall[n:], z[n:]
        beta, *_ = np.linalg.lstsq(X, z[:n], rcond=None)
        sig = np.sqrt(np.mean((z[:n] - X @ beta) ** 2))
        lo, hi = Xs @ beta - 1.645 * sig, Xs @ beta + 1.645 * sig
        pr = nig_predictive(Xs, *nig_posterior(X, z[:n], m0, V0, a0, b0))
        blo, bhi = pr.ppf(0.05), pr.ppf(0.95)
        cov_plug.append(np.mean((yf >= lo) & (yf <= hi))); cov_bayes.append(np.mean((yf >= blo) & (yf <= bhi)))
        w_plug.append(np.mean(hi - lo)); w_bayes.append(np.mean(bhi - blo))
    rows.append({"n (months)": n, "plug-in coverage": np.mean(cov_plug), "Bayes coverage": np.mean(cov_bayes),
                 "plug-in width": np.mean(w_plug), "Bayes width": np.mean(w_bayes)})
pd.DataFrame(rows).set_index("n (months)")

,plug-in coverage,Bayes coverage,plug-in width,Bayes width
n (months),,,,
18,0.397,0.897,0.081,0.286
24,0.572,0.903,0.110,0.244
36,0.708,0.902,0.131,0.209
60,0.799,0.902,0.144,0.188
120,0.850,0.895,0.154,0.175


With 1.5–2 years of monthly data, plug-in 90% intervals cover far less than 90%. The Bayesian predictive is close
to nominal at every $n$. (With this vague prior it coincides with the classical OLS *prediction* interval with
leverage and $t_{n-p}$ quantiles, the frequentist way of including parameter uncertainty.) The gap disappears as $n$ grows: epistemic uncertainty shrinks like $1/n$, while the noise does not.

## 2. Bayesian AR model for AirPassengers: posterior predictive simulation
Model on $z_t=\log y_t$: $z_t=c+\delta t+\text{month}_t+\phi z_{t-1}+\varepsilon_t$.
Multi-step forecasts need **simulation**: draw $(\beta,\sigma^2)$ from the posterior, then roll the AR recursion
forward with fresh noise. Each draw gives one path, and the collection of paths *is* the posterior predictive.

In [3]:
air = pf.load_air_passengers()
z = np.log(air)
train, test = z[:"1959-12-01"], z["1960-01-01":]
D = pf.seasonal_design(z.index, "1949-01-01")
D["lag1"] = z.shift(1)
Xtr, ytr = D.loc[train.index].iloc[1:].to_numpy(), train.iloc[1:].to_numpy()
Xfut = D.loc[test.index].drop(columns="lag1").to_numpy()
k = Xtr.shape[1]
post = nig_posterior(Xtr, ytr, np.zeros(k), np.eye(k) * 1e8, -k / 2, 0.0)       # reference prior, as in §1
mn, Vn, an, bn = post

def posterior_draws(n_draws):
    sig2 = 1 / rng.gamma(an, 1 / bn, n_draws)
    betas = np.array([rng.multivariate_normal(mn, s2 * Vn) for s2 in sig2])
    return betas, sig2

def simulate(betas, sig2, n_paths_per_draw):
    H = len(Xfut)
    out = np.empty((len(betas), n_paths_per_draw, H))
    for d, (b, s2) in enumerate(zip(betas, sig2)):
        zprev = np.full(n_paths_per_draw, train.iloc[-1])
        for h in range(H):
            zprev = Xfut[h] @ b[:-1] + b[-1] * zprev + rng.normal(0, np.sqrt(s2), n_paths_per_draw)
            out[d, :, h] = zprev
    return out                                  # (draws, paths, H) on the log scale

betas, sig2 = posterior_draws(400)
sims = simulate(betas, sig2, 25)
print(f"posterior mean of phi = {mn[-1]:.3f}, 90% CrI = [{np.quantile(betas[:, -1], .05):.3f}, {np.quantile(betas[:, -1], .95):.3f}]")

# plug-in comparison: fix beta, sigma at the posterior mean
b_hat, s2_hat = mn, bn / (an - 1)
plug = simulate(b_hat[None, :], np.array([s2_hat]), 10_000)[0]

posterior mean of phi = 0.792, 90% CrI = [0.694, 0.884]


In [4]:
bayes_paths = np.exp(sims.reshape(-1, len(Xfut)))
plug_paths = np.exp(plug)
taus = [0.05, 0.25, 0.5, 0.75, 0.95]
PANELS = [("Bayesian posterior predictive", bayes_paths), ("Plug-in (θ fixed)", plug_paths)]
fig = pf.subplots(1, 2, shared_yaxes=True, height=440,
                  titles=[f"{name}: CRPS = {pf.crps_sample(np.exp(test.values), pth).mean():.2f}" for name, pth in PANELS])
for c, (name, pth) in enumerate(PANELS, start=1):
    pf.add_fan(fig, test.index, {t: np.quantile(pth, t, axis=0) for t in taus}, history=air[:"1959-12-01"],
               actual=np.exp(test.values), n_history=36, row=1, col=c, showlegend=c == 1, fmt=",.0f")
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.12, x=0))
fig.show()

With 11 years of data the two fans are nearly identical, and CRPS on a single year cannot separate them. This matches
§1: parameter uncertainty matters most for *short* histories. It still accounts for a noticeable share of the variance:

### Decomposing the predictive variance by horizon
For each posterior draw $d$, the variance across its own paths is aleatoric. The variance of the draw-means is epistemic.

In [5]:
aleatoric = sims.var(axis=1).mean(axis=0)              # E_theta[Var(y|theta)]
epistemic = sims.mean(axis=1).var(axis=0)              # Var_theta(E[y|theta])
h = np.arange(1, len(Xfut) + 1)
fig = go.Figure()
fig.add_trace(go.Bar(x=h, y=aleatoric, name="aleatoric (noise)", marker_color=pf.SERIES[0],
                     hovertemplate="aleatoric %{y:.5f}<extra></extra>"))
fig.add_trace(go.Bar(x=h, y=epistemic, name="epistemic (parameters)", marker_color=pf.SERIES[1],
                     hovertemplate="epistemic %{y:.5f}<extra></extra>"))
fig.update_layout(title="Posterior predictive variance decomposition", barmode="stack", height=380, hovermode="x unified",
                  xaxis_title="horizon (months)", yaxis_title="variance of log y")
fig.show()
print("epistemic share by horizon:", np.round(epistemic / (aleatoric + epistemic), 2))

epistemic share by horizon: [0.16 0.17 0.2  0.21 0.21 0.22 0.23 0.24 0.25 0.26 0.26 0.27]


## 3. Gaussian-process forecast (NumPy implementation)
A GP is a prior over functions $f\sim\mathcal{GP}(0,k)$ with $y=f(t)+\varepsilon$. The kernel encodes our beliefs:
$$k(t,t')=\underbrace{s_p^2\exp\!\Big(-\tfrac{2\sin^2(\pi|t-t'|/12)}{\ell_p^2}\Big)\exp\!\Big(-\tfrac{(t-t')^2}{2\ell_d^2}\Big)}_{\text{yearly pattern that slowly evolves}}+\underbrace{s_r^2\exp\!\Big(-\tfrac{(t-t')^2}{2\ell_r^2}\Big)}_{\text{smooth deviations from the trend}}+\sigma_n^2\delta_{tt'}$$
We remove a linear trend first (GPs revert to their mean when extrapolating), fit the hyper-parameters by maximising
the **log marginal likelihood**, and get a closed-form Gaussian predictive:
$\mu_*=K_{*}^{\top}K^{-1}y$, $\Sigma_*=K_{**}-K_*^\top K^{-1}K_*$.

In [6]:
t_tr = np.arange(len(train), dtype=float)
t_te = np.arange(len(train), len(z), dtype=float)
coef = np.polyfit(t_tr, train.values, 1)
r_tr = train.values - np.polyval(coef, t_tr)

def kernel(a, b, th):
    lp, ld, sp, lr, sr = np.exp(th[:5])
    d = a[:, None] - b[None, :]
    return (sp**2 * np.exp(-2 * np.sin(np.pi * d / 12) ** 2 / lp**2) * np.exp(-d**2 / (2 * ld**2))
            + sr**2 * np.exp(-d**2 / (2 * lr**2)))

def neg_log_ml(th):
    K = kernel(t_tr, t_tr, th) + (np.exp(th[5]) ** 2 + 1e-8) * np.eye(len(t_tr))
    try:
        L = np.linalg.cholesky(K)
    except np.linalg.LinAlgError:
        return 1e10
    a = np.linalg.solve(L.T, np.linalg.solve(L, r_tr))
    return 0.5 * r_tr @ a + np.log(np.diag(L)).sum() + 0.5 * len(r_tr) * np.log(2 * np.pi)

th0 = np.log([1.0, 60.0, 0.1, 12.0, 0.05, 0.03])
opt = optimize.minimize(neg_log_ml, th0, method="L-BFGS-B", bounds=[(-3, 3), (1, 7), (-6, 1), (0, 5), (-6, 1), (-7, -1)])
th = opt.x
print("hyper-parameters [l_p, l_d, s_p, l_r, s_r, sigma_n] =", np.round(np.exp(th), 3))

K = kernel(t_tr, t_tr, th) + np.exp(th[5]) ** 2 * np.eye(len(t_tr))
Ks, Kss = kernel(t_tr, t_te, th), kernel(t_te, t_te, th)
L = np.linalg.cholesky(K)
alpha_ = np.linalg.solve(L.T, np.linalg.solve(L, r_tr))
v = np.linalg.solve(L, Ks)
gp_mu = Ks.T @ alpha_ + np.polyval(coef, t_te)
gp_cov = Kss - v.T @ v + np.exp(th[5]) ** 2 * np.eye(len(t_te))
gp_sd = np.sqrt(np.diag(gp_cov))

gp_paths = np.exp(rng.multivariate_normal(gp_mu, gp_cov, 4000))      # joint GP predictive -> paths
fig = pf.add_fan(go.Figure(), test.index, {t: np.exp(gp_mu + gp_sd * stats.norm.ppf(t)) for t in taus},
                 history=air[:"1959-12-01"], actual=np.exp(test.values), n_history=36, color=pf.SERIES[2], label="GP", fmt=",.0f")
fig.update_layout(title=f"Gaussian process forecast: CRPS = {pf.crps_sample(np.exp(test.values), gp_paths).mean():.2f}",
                  yaxis_title="passengers (thousands)")
fig.show()

hyper-parameters [l_p, l_d, s_p, l_r, s_r, sigma_n] = [5.91000e-01 2.34664e+02 1.36000e-01 7.46900e+00 4.50000e-02 2.50000e-02]


## 4. What a full probabilistic-programming model looks like
For richer models (hierarchical seasonality across many SKUs, change-points, count likelihoods) use PyMC / NumPyro / Stan.
A sketch in PyMC (not executed here; needs `pip install pymc`):

```python
import pymc as pm
with pm.Model() as m:
    level   = pm.Normal("level", 5, 1)
    trend   = pm.Normal("trend", 0, 0.05)
    season  = pm.ZeroSumNormal("season", sigma=0.3, shape=12)        # month effects sum to zero
    sigma   = pm.HalfNormal("sigma", 0.2)
    nu      = pm.Gamma("nu", 2, 0.1)                                  # Student-t tails
    mu = level + trend * t + season[month_idx]
    pm.StudentT("y", nu=nu, mu=mu, sigma=sigma, observed=log_y)
    idata = pm.sample()                                                # NUTS
    # posterior predictive for new months: pm.set_data(...) then pm.sample_posterior_predictive(idata)
```

## 5. Takeaways
* The posterior predictive automatically includes parameter uncertainty. This matters most for **short histories**,
  **many parameters**, and **long horizons** (trend extrapolation).
* Priors are a feature: they regularise (e.g. shrink month effects) and allow **pooling** across related series.
* Bayesian output is a set of samples, so every tool from nb 05 (paths, functionals, CRPS, rank histograms) applies.
* Check calibration in a backtest anyway. A wrong *likelihood* is not fixed by being Bayesian.

### Exercises
1. In §1, use an informative prior $N(0, 0.1^2)$ on the month effects with $n=18$. How do coverage and width change?
2. Add a Student-t likelihood via data augmentation or PyMC and compare tail coverage.
3. Add a linear kernel to the GP instead of detrending first, so that trend uncertainty is included.